## Given this code, what can you change in order to improve the performance here?

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # You may add normalization to improve the performance
])

# Data
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# Model
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(), # You may use ReLU instead of Sigmoid
            nn.Linear(256, 128),
            nn.ReLU(), # You may use ReLU instead of Sigmoid
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.layers(x)

# Train and val
def train_and_evaluate(epochs=10):
    model = SimpleNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001) # You may change the optimizer to Adam and adjust the learning rate as well

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total

        # Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_acc = 100 * correct / total
        print(f"Epoch {epoch+1} | Loss: {running_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

train_and_evaluate(epochs=10)

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 495kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.97MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.7MB/s]


Epoch 1 | Loss: 1055.799 | Train Acc: 21.97% | Test Acc: 35.45%
Epoch 2 | Loss: 1000.642 | Train Acc: 46.09% | Test Acc: 56.45%
Epoch 3 | Loss: 920.277 | Train Acc: 61.45% | Test Acc: 66.80%
Epoch 4 | Loss: 805.656 | Train Acc: 68.47% | Test Acc: 71.69%
Epoch 5 | Loss: 672.656 | Train Acc: 72.92% | Test Acc: 75.53%
Epoch 6 | Loss: 555.157 | Train Acc: 76.61% | Test Acc: 79.28%
Epoch 7 | Loss: 467.449 | Train Acc: 79.51% | Test Acc: 81.44%
Epoch 8 | Loss: 404.189 | Train Acc: 81.57% | Test Acc: 82.81%
Epoch 9 | Loss: 357.504 | Train Acc: 83.00% | Test Acc: 84.03%
Epoch 10 | Loss: 322.096 | Train Acc: 84.06% | Test Acc: 84.89%
